# Portfolio Risk Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Present Euler VaR/ES contributions and risk-budget diagnostics for portfolio risk review.

**Prerequisites:** `05_portfolio/portfolio_risk_decomposition.ipynb`.

**What you'll learn:**

- Prepare risk-decomposition inputs.
- Render `reporting.portfolio_risk_tearsheet`.
- Connect contribution analytics to a risk-budget narrative.

Decompose portfolio VaR/ES into per-position Euler contributions, evaluate a risk
budget, and render a `portfolio_risk_tearsheet`.


In [ ]:
import datetime as dt

from finstack_quant import reporting
from finstack_quant.models.factor.risk import (
    evaluate_risk_budget,
    parametric_es_decomposition,
    parametric_var_decomposition,
)

position_ids = ["Equity", "Credit", "Rates"]
weights = [0.50, 0.30, 0.20]
covariance = [
    [0.0400, 0.0120, -0.0020],
    [0.0120, 0.0200, 0.0010],
    [-0.0020, 0.0010, 0.0064],
]
decomposition = parametric_var_decomposition(position_ids, weights, covariance, 0.95)
es = parametric_es_decomposition(position_ids, weights, covariance, 0.95)
actual = [c.component_var for c in decomposition.var_contributions]
budget = evaluate_risk_budget(position_ids, actual, [0.40, 0.35, 0.25], decomposition.portfolio_var, 1.10)
print("Portfolio VaR:", round(decomposition.portfolio_var, 4), " breach:", budget.has_breach)

## Risk decomposition & budget

In [ ]:
reporting.portfolio_risk_tearsheet(
    decomposition, es=es, budget=budget, title="Sleeve Risk", generated=dt.date(2026, 6, 23)
)

## Saving a standalone HTML file

```python
ts = reporting.portfolio_risk_tearsheet(decomposition, es=es, budget=budget, generated=dt.date(2026, 6, 23))
ts.save("portfolio_risk_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
